In [ ]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

from simtk.openmm.app import *
from simtk.openmm import *
from simtk.unit import *
import PeptideBuilder.Geometry
from PeptideBuilder import Geometry
import pdbfixer
from pdbfixer import PDBFixer
from mixture_maker import *
import MDAnalysis as mda
from collections import Counter
import PeptideConstructor as pc
from parmed import load_file
import Bio.PDB
import math
from PeptideConstructor import Geometry
from PeptideConstructor import PeptideBuilder

# Make PDB files of sequences in list

## Manual sequence : ratio list

### PEPTIDE WITH GLYCINE AS CENTRAL RESIDUE SHOULD ALWAYS BE FIRST IN THIS LIST!!!

### Automated sequence : ratio list (changes ratios of different residue groupings)

In [ ]:
i = '/Volumes/potoyan-lab/will/hnrnpa1/7bx7-monomer_fixed.pdb'
j = '/Volumes/potoyan-lab/will/hnrnpa1/7bx7-monomer_fixed_2.pdb'

from pdbfixer import PDBFixer
from simtk.openmm.app import PDBFile
fixer = PDBFixer(filename = i)
fixer.findMissingResidues()
fixer.findMissingAtoms()
fixer.addMissingAtoms()
fixer.addMissingHydrogens(7.0)
PDBFile.writeFile(fixer.topology, fixer.positions, open(j, 'w'))

In [ ]:
seq_str = 'GSMASASSSQRGRSGSGNFGGGRGGGFGGNDNFGRGGNFSGRGGFGGSRGGGGYGGSGDGYNGFGNDGSNFGGGGSYNDFGNYNNQSSNFGPMKGGNFGGRSSGPYGGGGQYFAKPRNQGGYGGSSSSSSYGSGRRF'

seq_num_list = [(seq_str, 1)]





In [ ]:
chain_length = 9

res_list = ['G','A','R','N','D','C','E','Q','H','I','L','K','M','F','P','S','T','W','Y','V']

'''
GXGGGGGXG
GGGXGXGGG
GXGGXGGXG
GGGXXXGGG
'''
pdbs = []

for res in res_list:
    pdbs.append('GGGG'+f'{res}'+'GGGG')


In [ ]:
charged = ['R','K','H','D','E']
aromatic = ['F','Y','W']
polar = ['S','T','N','Q','C']
nonpolar = ['G','P','A','V','I','L','M']

fname = 'equal'

middle = int((chain_length-1)/2)

ratio = '1:1'

seq_num_list = []

for file in pdbs:
    
    pep = file[0:chain_length]
    
    if fname == 'charged':
        if pep[middle] in charged:
            num = int(ratio[0])
        else:
            num = int(ratio[2])
    elif fname == 'aromatic':
        if pep[middle] in aromatic:
            num = int(ratio[0])
        else:
            num = int(ratio[2])
    elif fname == 'polar':
        if pep[middle] in polar:
            num = int(ratio[0])
        else:
            num = int(ratio[2])
    elif fname == 'nonpolar':
        if pep[middle] in nonpolar:
            num = int(ratio[0])
        else:
            num = int(ratio[2])
    else:
        num = int(ratio[2])
            
    seq_num_list.append((pep, num))


In [ ]:
seq_num_list

# Create Packmol .inp File

In [ ]:
# fname='test'

# path = f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/9_res/{fname}/'

path='./'

# run_packmol(seq_num_list, terminal_type='charged', path=path, d=100, l=120)


# Run Packmol

In [ ]:
!cat /Volumes/potoyan-lab/will/gxg-project/notebooks/mixture.inp #packmol input script

!packmol < /Volumes/potoyan-lab/will/gxg-project/notebooks/mixture.inp

# !mv ./packed.pdb {path}

# !rm ./*.pdb*

In [ ]:
import nglview as nv
import MDAnalysis as mda

u = mda.Universe('/Volumes/potoyan-lab/will/hnrnpa1/75mgml/asn_ser/packed.pdb')

view = nv.show_mdanalysis(u)
# view.add_ball_and_stick()
view.add_unitcell()
view

# OpenMM Solvation

In [ ]:
# Use this solvation and use manually edited "a99SBdisp_tip4pd.ff" 
# after solvating with "tip4pew" water model below


In [ ]:
# Make sure O5 terminal residues are recognized for dna/rna, 
# delete a few lines before or after if needed.

path = '/Volumes/potoyan-lab/will/hnrnpa1/75mgml/asn_ser/'

pdb        = PDBFile(f'{path}packed.pdb')

modeller   = Modeller(pdb.topology, pdb.positions)
modeller.addHydrogens()

forcefield = ForceField('amber14-all.xml', 'amber14/tip4pew.xml')
modeller.addSolvent(forcefield, model='tip4pew',  boxSize=Vec3(100.0, 100.0, 100.0)*angstroms, ionicStrength=0.1*molar)

system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME)
integrator = VerletIntegrator(0.001*picoseconds)
simulation = Simulation(modeller.topology, system, integrator)
simulation.context.setPositions(modeller.positions)
simulation.minimizeEnergy(maxIterations=100)

print('Saving PDB...')
positions = simulation.context.getState(getPositions=True).getPositions()
PDBFile.writeFile(simulation.topology, positions, open(f'{path}output.pdb', 'w'))


In [ ]:
u = mda.Universe('/Volumes/potoyan-lab/will/hnrnpa1/75mgml/asn_ser/output.pdb')

view = nv.show_mdanalysis(u)
view.add_ball_and_stick()
view.add_unitcell()
view

###                                                    ###
### NEEDS TO BE DONE FOR FORCEFIELDS WITH NO .XML FILE ###
###                                                    ###

### Need to be run in terminal to generate gro and top files + itps ###

!gmx pdb2gmx -f output.pdb -o output.gro -p output.top -ignh


In [ ]:
# for system in ['50mgml','75mgml','100mgml']:
path = f'/Volumes/potoyan-lab/will/hnrnpa1/75mgml/asn_ser/'
top = load_file(f'{path}output.top')
top.save(f'{path}output.top', overwrite=True)
!rm {path}*.itp*


# Calculate density of system

In [ ]:
from Bio.SeqUtils import seq1
from Bio.SeqUtils import molecular_weight

def listtostring(s):
    str1 = ""
    for ele in s:
        str1 += ele
    return str1

def rnalisttostring(s):
    str1 = ""
    for ele in s:
        str1 += ele[1]
    return str1

path = '/Volumes/potoyan-lab/will/hnrnpa1/75mgml/asn_ser/'

u = mda.Universe(f'{path}output.pdb')
prot = u.select_atoms('protein')
rna = u.select_atoms('nucleic')

system_list = prot.residues.resnames.tolist()
box_str = listtostring(system_list)
new_box_str = seq1(box_str)
box_mass_amu = molecular_weight(new_box_str,'protein')

system_nucleic_list = rna.residues.resnames.tolist()
box_nucleic_str = rnalisttostring(system_nucleic_list)
box_nucleic_mass_amu = molecular_weight(box_nucleic_str,'DNA')

box_mass_mg = (box_mass_amu + box_nucleic_mass_amu) / 6.023e20

box_volume_ml = (u.dimensions[0]*u.dimensions[1]*u.dimensions[2]) / (1.0E24)

box_mass_mg / box_volume_ml




# Short Trial OpenMM Sim

In [ ]:
from simtk.openmm.app import *
from simtk.openmm import *
from simtk.unit import *
from sys import stdout
from parmed import load_file

gro = load_file('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/equal/output.gro')
top = load_file('/Volumes/potoyan-lab/will/gxg-project/gxg_comps/equal/output.top')#, 
#                      periodicBoxVectors=gro.get_box(),
#                      includeDir='/usr/local/gromacs/share/gromacs/top')
                
top.box = gro.box[:]

system = top.createSystem(nonbondedMethod=PME, nonbondedCutoff=1*nanometer,
        constraints=HBonds)
integrator = LangevinIntegrator(300*kelvin, 1/picosecond, 0.002*picoseconds)
simulation = Simulation(top.topology, system, integrator)
simulation.context.setPositions(gro.positions)
simulation.minimizeEnergy()
simulation.reporters.append(PDBReporter('50_capped_output_2.pdb', 1000))
simulation.reporters.append(StateDataReporter(stdout, 1000, step=True,
        potentialEnergy=True, temperature=True))
simulation.step(10000)